In [ ]:
import sys, os
sys.path.append(os.path.join("..", "src"))

import numpy as np
import matplotlib.pyplot as plt
import chirps, filters, utils

rng = np.random.default_rng(42)

fs = 20e6
f0, f1 = 1e6, 5e6
dur = 20e-6
taper = 0.08

plt.rcParams["figure.figsize"] = (10, 4)

In [ ]:
"""base chirp diagrams"""
t, chirp = chirps.linear_chirp(f0, f1, dur, fs, taper)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(t * 1e6, np.real(chirp))
ax1.set_xlabel("Time (µs)")
ax1.set_ylabel("Amplitude")
ax1.set_title("Chirp (time domain)")

ax2.specgram(np.real(chirp), NFFT=64, noverlap=48, Fs=fs)
ax2.set_xlabel("Time (s)")
ax2.set_ylabel("Frequency (Hz)")
ax2.set_title("Spectrogram")

fig.savefig("../figures/01_chirp.png", dpi=150, bbox_inches="tight")
plt.tight_layout()

In [ ]:
"""unwindowed vs tukey window specgram"""

_, x_rect = chirps.linear_chirp(f0, f1, dur, fs, taper=0.0)
_, x_tuk  = chirps.linear_chirp(f0, f1, dur, fs, taper=taper)

fr, db_rect = chirps.spectrum(np.real(x_rect), fs)
ft, db_tuk  = chirps.spectrum(np.real(x_tuk),  fs)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.plot(t * 1e6, x_rect, label="rectangular")
ax1.plot(t * 1e6, x_tuk,  label=f"Tukey ({taper:g})")
ax1.set(xlabel="time (µs)", ylabel="amplitude", title="time domain")
ax1.legend()

ax2.plot(fr / 1e6, db_rect, label="rectangular")
ax2.plot(ft / 1e6, db_tuk,  label=f"Tukey ({taper:g})")
ax2.set(xlabel="frequency (MHz)", ylabel="dB", title="spectrum",
        xlim=(0, 8), ylim=(-80, 5))
ax2.legend()

fig.savefig("../figures/02_windowing.png", dpi=150, bbox_inches="tight")
fig.tight_layout()

In [ ]:
"""pulse compression demo"""
ref = np.real(chirp)
echo = utils.make_echo(ref, [1500], [1.0])
y = filters.wiener_filter(echo, ref)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 6))

ax1.plot(echo)
ax1.set(xlabel="sample", ylabel="amplitude",
        title="received echo (the chirp, spread out in time)")

ax2.plot(utils.envelope_db(y))
ax2.set(xlabel="sample", ylabel="dB", ylim=(-60, 2),
        title="after matched filter (compressed to a peak)")

fig.savefig("../figures/03_compression.png", dpi=150, bbox_inches="tight")
fig.tight_layout()

print(f"peak at sample: {np.argmax(np.abs(y))}  (scatterer was at 1500)")
print(f"-6 dB main-lobe width: {utils.mainlobe_width(y)} samples")
print(f"peak sidelobe: {utils.peak_sidelobe_db(y):.1f} dB")

In [ ]:
"""matched vs naive inverse vs Wiener"""
ref = np.real(x_tuk)
echo = utils.add_noise(utils.make_echo(ref, [1500], [1.0]), 15, rng)

y_mf   = filters.matched_filter(echo, ref)
y_inv  = filters.naive_inverse_filter(echo, ref)
y_wien = filters.wiener_filter(echo, ref, gamma=1.0)
curves = [(y_mf, "matched"), (y_inv, "naive inverse"), (y_wien, "Wiener (γ=1)")]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

for y, name in curves:
    ax1.plot(utils.envelope_db(y), label=name, alpha=0.8)
ax1.set(xlabel="sample", ylabel="dB", ylim=(-60, 2),
        title="full record — inverse amplifies noise everywhere")
ax1.legend()

lo, hi = 1440, 1560
for y, name in curves:
    ax2.plot(range(lo, hi), utils.envelope_db(y)[lo:hi], label=name, alpha=0.8)
ax2.axhline(-6, ls="--", c="gray", lw=0.8)
ax2.set(xlabel="sample", ylabel="dB", ylim=(-40, 2),
        title="zoom on main lobe — resolution vs sidelobes")
ax2.legend()

fig.savefig("../figures/04_three_filters.png", dpi=150, bbox_inches="tight")
fig.tight_layout()

for y, name in curves:
    print(f"{name:14s}  width {utils.mainlobe_width(y):3d}   sidelobe {utils.peak_sidelobe_db(y):6.1f} dB")

In [ ]:
"""gamma sweep: resolution / sidelobe / SNR tradeoff"""
ref = np.real(x_tuk)                     
gammas = np.logspace(-3, 3, 25)
snr_levels = [5, 15, 25]                   # dB, noisy -> clean

def peak_snr_db(y, noise_region=slice(0, 1000)):
    pk = np.max(np.abs(y))
    nf = np.sqrt(np.mean(np.abs(y[noise_region])**2))
    return 20 * np.log10(pk / (nf + 1e-12))

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
for snr in snr_levels:
    echo = utils.add_noise(utils.make_echo(ref, [1500], [1.0]), snr, rng)
    width, slobe, osnr = [], [], []
    for g in gammas:
        y = filters.wiener_filter(echo, ref, gamma=g)
        width.append(utils.mainlobe_width(y))
        slobe.append(utils.peak_sidelobe_db(y))
        osnr.append(peak_snr_db(y))
    axes[0].semilogx(gammas, width, marker=".", label=f"{snr} dB")
    axes[1].semilogx(gammas, slobe, marker=".", label=f"{snr} dB")
    axes[2].semilogx(gammas, osnr,  marker=".", label=f"{snr} dB")

axes[0].set(xlabel="γ", ylabel="samples", title="main-lobe width (resolution)")
axes[1].set(xlabel="γ", ylabel="dB",      title="peak sidelobe level")
axes[2].set(xlabel="γ", ylabel="dB",      title="output peak SNR")
for ax in axes:
    ax.legend(title="echo SNR"); ax.grid(alpha=0.3)
fig.suptitle("γ sweep - wiener tradeoff across noise levels")
fig.savefig("../figures/05_gamma_sweep.png", dpi=150, bbox_inches="tight")
fig.tight_layout()